

Step	学習内容	理解すること
1	Motion Subspace	関節運動を6次元Spatial Vectorで表す
2	Spatial Velocityの再帰計算	親リンクから子リンクへ速度を伝播
3	Spatial Accelerationの再帰計算	関節加速度とコリオリ項を考慮
4	Spatial Forceの計算	Spatial Newton-Euler式を利用
5	RNEAのBackward Recursion	子リンクから親リンクへ力を伝播し、関節トルクを計算
6	2リンクアームで検証	以前Kane法で導出した運動方程式と比較

## 1. Motion Subspace

Spatial Velocityは角速度と並進速度を用いて以下である。

$$
\boldsymbol{V} = \begin{bmatrix} \boldsymbol{\omega} \\ \boldsymbol{v} \end{bmatrix} \in \mathbf{R}^{6 \times 1}
$$

以下の親リンクにたいする関節の相対運動として、$z_i$軸に回転自由度がある剛体の Spatial Velocity を考える。

<p align="center">
<img src="./images/1DoF_revolute_joint.png" style="width:34%;" />
</p>

この場合、基準点を関節部の$J$とすれば、角速度 $\boldsymbol{\omega}$ と並進速度 $\boldsymbol{v}_J$ は以下となる。

$$
\boldsymbol{\omega} = \begin{bmatrix} 0 \\ 0 \\ \dot{q} \end{bmatrix}, \quad \boldsymbol{v}_J = \begin{bmatrix} 0 \\ 0 \\ 0 \end{bmatrix}
$$

よって、Spatial Velocityは以下となる。

$$
\boldsymbol{V}_J = \begin{bmatrix} \boldsymbol{\omega} \\ \boldsymbol{v}_J \end{bmatrix} =
\begin{bmatrix} 0 \\ 0 \\ \dot{q} \\ 0 \\ 0 \\ 0 \end{bmatrix} =
\underbrace{\begin{bmatrix} 0 \\ 0 \\ 1 \\ 0 \\ 0 \\ 0 \end{bmatrix}}_{\boldsymbol{S}} \dot{q}
$$

上式の$\boldsymbol{S}$を Motion Subspace と呼び、関節の一般化速度をSpatial Velicityに変換する写像と理解できる。

$$
\boldsymbol{V}_J = \boldsymbol{S} \dot{q}
$$


### RNEA とSubspaceとの関連

RNEAでは各リンクのSpatial Velocityを次のように計算する。

$$
\boldsymbol{V}_i = {^i}X_{\lambda(i)} \ \boldsymbol{V}_{\lambda(i)} + \boldsymbol{S}_i \dot{q}_i
$$

ここで、

- $\lambda(i)$ : リンク$i$の親リンク
- ${^i}X_{\lambda(i)}$ : 親リンク$\lambda(i)$のSpatial Velocityを子リンク$i$座標系へ変換する行列
- $\boldsymbol{S}_i \dot{q}_i$ : 親リンクに対する相対運動

さらにSpatial Force Vector $\boldsymbol{f}_i$ から以下のように関節トルクをSubspace で取り出すことが出来る。

$$
\tau_i = \boldsymbol{S}_i^T \boldsymbol{f}_i
$$

この場合、Motion Subspaceは関節の一般化速度からSpatial Velocityを作るだけでなく、Spatial Force Vector から関節の一般化力を取り出す役割も持つ。

これは以下のように導くことが出来る。

Spatial Power(仕事率)は以下で計算できる。

$$
P = \boldsymbol{f}^T \boldsymbol{V}
$$

関節の相対速度を$\boldsymbol{V} = S \dot{q}$ とすると、以下となる。

$$
P = \boldsymbol{f}^T S \dot{q} = (S^T \boldsymbol{f}) \dot{q}
$$

一方、一般化力$\tau$と一般化速度$\dot{q}$により仕事率は以下である。

$$
P = \tau^T \dot{q}
$$

したがって、任意の$\dot{q}$に対して仕事率が一致するためには、以下の関係が必要である。

$$
\tau = S^T \boldsymbol{f}
$$


## 2. Spatial Velocityの再帰計算(Forward Recursion)

### 2.1 親リンクと子リンクの速度の関係

親リンクを$i-1$、子リンクを$i$とする。

子リンクの運動は次の２つから構成される。

1. 親リンクの運動によって生じる速度
2. 親リンクに対する子リンクの相対運動によって生じる速度

よって子リンクのSpatial Velocityは次のように計算できる。

$$
{^i}V_{i,O_i} = {^i}X_{i-1, O_{i-1} \rightarrow O_i} \ {}^{i-1}\boldsymbol{V}_{i-1, O_{i-1}} + {^i}\boldsymbol{S}_{i, O_i} \dot{q}_i
$$

各項の意味は以下となる。

- ${^i}V_{i,O_i}$ : 子リンク$i$の$i$座標系の基準点$O_i$におけるSpatial Velocity
- ${^i}X_{i-1, O_{i-1} \rightarrow O_i}$ : 親リンクのSpatial Velocityを子リンクの座標系へ変換し、基準点を$O_{i-1}$から$O_i$へ変更するSpatial Transform
- ${i-1}\boldsymbol{V}_{i-1, O_{i-1}}$ : 親リンク$i-1$の$i-1$座標系の基準点$O_{i-1}$におけるSpatial Velocity
- ${^i}\boldsymbol{S}_{i, O_i} \dot{q}_i$ : 親リンク$i-1$に対する子リンク$i$の基準点$O_i$における相対Spatial Velocity

RNEAでは各リンクに剛体座標系を設定し、Spatial Velocityをその座標系の原点で表す約束がある。<br>
また、Motion Subspaceにも基準点が必要であり、例えばリンク$i$の関節がz軸で回転し、その軸が$O_i$を通るなら、Motion Subspaceは以下となる。

$$
{^i}\boldsymbol{S}_{i, O_i} = \begin{bmatrix} 0 & 0 & 1 & 0 & 0 & 0 \end{bmatrix}^T
$$

しかし、基準点$P$が関節軸$O_i$から離れている場合、以下のようになる。

$$
{^i}\boldsymbol{S}_{i, P} = \begin{bmatrix} e_z \\ e_z \times {^i}\boldsymbol{r}_{O_i P} \end{bmatrix}^T
$$

よって、座標系とその原点は次のように設定を行う。

- リンクが回転する場合、回転する軸に座標系のある軸を合わせるように設定する。原点は合わせた軸上となるが、その位置は親リンクの原点から適切な位置とする。
- リンクが直動する場合、並進する方向に座標系のある軸を合わせるように設定する。原点は合わせた軸上となるが、その位置は親リンクの原点から適切な位置とする。

例えば、親リンク$i-1$と子リンク$i$が下の図のように構成されており、親リンクは$v_{i-1}$方向に並進し、子リンクは$O_i$まわりに回転しているとする。親座標系は$v_{i-1}$と$x_{i-1}$が一致するように設定し、その原点は剛体中心としている。子座標系は回転軸と$z_i$方向を一致するように設定し、原点を回転軸上としている。

<p align="center">
<img src="./images/rnea-1.png" style="width:40%;" />
</p>

RNEAでは、それぞれのリンクの運動の基準点をそのリンクの座標原点で示す約束となるため、今後は上式から基準点の情報を省略した以下の記述を行う。

改めて、子リンクのSpatial Velocityの計算式は以下と再帰式となる。

$$
{^i}\boldsymbol{V}_{i} = {^i}X_{i-1} \ {^{i-1}}\boldsymbol{V}_{i-1} + {^i}\boldsymbol{S}_{i} \dot{q}_i
$$

- ${^i}\boldsymbol{V}_{i}$ : リンク$i$座標で示したリンク$i$のSpatial Velocity
- ${^i}X_{i-1}$ : 親リンクのSpatial Velocityを子リンクの座標系・基準点へ変換するSpatial Transform
- ${^i}\boldsymbol{S}_{i} \dot{q}_i$ : 親リンクに対する子リンクの相対 Spatial Velocity

### 2.2 Spatial Transform の展開

[このNote](./spatial_vector.ipynb)の2節でSpatial VelocityのSpatial Transformを導いた。RENAでは親リンクの原点$O_{i-1}$から子リンクの原点$O_i$への基準点変換を行う。$O_{i-1}$から$O_i$までの位置ベクトルを子座標で表したものを以下とする。

$$
{}^i \boldsymbol{r}_{i-1, i}
$$

Spatial Transformは以下となる。

$$
{^i}X_{i-1} = \begin{bmatrix}
{}^i R _{i-1} & 0 \\
-[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1} & {}^i R _{i-1}
\end{bmatrix}
$$

再帰式に代入すると以下となる。

$$
\begin{bmatrix} {}^i \boldsymbol{\omega}_i \\ {}^i \boldsymbol{v}_i \end{bmatrix} =
\begin{bmatrix}
{}^i R _{i-1} & 0 \\
-[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1} & {}^i R _{i-1}
\end{bmatrix}
\begin{bmatrix} {}^{i-1} \boldsymbol{\omega}_{i-1} \\ {}^{i-1} \boldsymbol{v}_{i-1} \end{bmatrix} +
{}^i \boldsymbol{S}_i \dot{q}_i
$$

### 2.3 具体例

上で示した図の子リンクのSpatial Velocityを求めてみる。

子リンクは子リンク座標系の$z_i$軸まわりに$\omega_i$で回転するとしている。そのため、Motion Subspaceは以下となる。

$$
{}^i \boldsymbol{S}_i = \begin{bmatrix} 0 & 0 & 1 & 0 & 0 & 0 \end{bmatrix}^T
$$

これより、子リンクのSpatial Velcotyは以下となる。

$$
\begin{bmatrix} {}^i \boldsymbol{\omega}_i \\ {}^i \boldsymbol{v}_i \end{bmatrix} =
\begin{bmatrix}
{}^i R _{i-1} & 0 \\
-[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1} & {}^i R _{i-1}
\end{bmatrix}
\begin{bmatrix} {}^{i-1} \boldsymbol{\omega}_{i-1} \\ {}^{i-1} \boldsymbol{v}_{i-1} \end{bmatrix} +
\begin{bmatrix} 0 \\ 0 \\ \omega_i \\ 0 \\ 0 \\ 0 \end{bmatrix} 
$$

角速度と並進速度に分解する。

- 角速度

    $$
    {}^i \boldsymbol{\omega}_i = {}^i R _{i-1} {}^{i-1} \boldsymbol{\omega}_{i-1} + \begin{bmatrix} 0 \\ 0 \\ \omega_i \end{bmatrix} 
    $$

    この式は親リンクの角速度を子リンク座標へ変換したものと、相対角速度の合成であることが分かる。

- 並進速度

    $$
    \begin{aligned}
    {}^i \boldsymbol{v}_i &= -[{}^i \boldsymbol{r}_{i-1, i}]^\land \ {}^i R_{i-1}{}^{i-1} \boldsymbol{\omega}_{i-1} + {}^i R _{i-1}{}^{i-1} \boldsymbol{v}_{i-1} \\
    &= {}^i R _{i-1}{}^{i-1} \boldsymbol{v}_{i-1} + ({}^i R_{i-1}{}^{i-1} \boldsymbol{\omega}_{i-1}) \times {}^i \boldsymbol{r}_{i-1, i}
    \end{aligned}
    $$

    ${}^i \boldsymbol{v}_i$は子リンク原点のSpatial Velocityである。よって、親リンクの並進速度と、親リンク原点と子リンク原点までの位置ベクトルを親リンクの角速度で回転させた接線速度との合成であることが分かる。
    
### 2.4 再帰計算について

例えば、3リンクアームの場合、固定ベースのSpatial Velocityを以下とする。

$$
{}^0 \boldsymbol{V}_0 = 0
$$

すると、2.1節で示したSpatial Velocityの再帰式より、リンク1からリンク2, リンク3へと順番に計算ができる。

$$
\begin{aligned}
{}^1 \boldsymbol{V}_1 =& {}^1 X_{0} \ {}^0 \boldsymbol{V}_0 + {}^1 \boldsymbol{S}_1 \dot{q}_1 = {}^1 \boldsymbol{S}_1 \dot{q}_1 \\
{}^2 \boldsymbol{V}_2 =& {}^2 X_{1} \ {}^1 \boldsymbol{V}_1 + {}^2 \boldsymbol{S}_2 \dot{q}_2 \\
{}^3 \boldsymbol{V}_3 =& {}^3 X_{2} \ {}^2 \boldsymbol{V}_2 + {}^3 \boldsymbol{S}_3 \dot{q}_3 \\
\end{aligned}
$$

各リンクの６次元ベクトルと$6 \times 6$ 行列の演算を行えばよく、全てのリンクの速度式の展開を求める必要がない。これがForward Recursionの利点である。



## 3. Spatial Accelerationの再帰計算

